# Lab 5 tasks

Name: Khair ul wara Hussain

Student Id: FA23-BAI-022

Course: NLP

---
# Lab 05: N-gram Language Models & Smoothing

## Setup: imports, dataset and helper functions

In [1]:
import os
import random
import warnings
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.util import ngrams
from nltk.lm import MLE, Laplace, Lidstone, KneserNeyInterpolated, WittenBellInterpolated

warnings.filterwarnings("ignore")
for pkg in ["punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

In [2]:
# product_reviews_corpus.txt -> one review sentence per line.
# If the file is already next to this notebook it is used as it is, otherwise a small
# sample corpus is written to disk so that the notebook can run from start to end.
if not os.path.exists("product_reviews_corpus.txt"):
    random.seed(42)
    subjects = ["the camera quality", "the battery life", "the screen", "the sound quality",
                "the build quality", "the price", "the delivery", "the packaging", "customer service",
                "the phone", "the laptop", "the headphones", "the keyboard", "the charger"]
    predicates = ["is excellent", "was better than i expected", "exceeded my expectations",
                  "is worth every penny", "stopped working after a week",
                  "is too small for me", "is a little disappointing", "is fantastic",
                  "is great for the price", "was very easy to set up", "feels cheap and flimsy",
                  "works perfectly", "is not as good as the description", "is very good"]
    extras = [
        "the camera quality exceeded my expectations", "the battery life exceeded my expectations",
        "the screen exceeded my expectations", "this phone exceeded my expectations",
        "the sound quality exceeded my expectations and the price was low",
        "the delivery was fast and the packaging was good",
        "i love this laptop and the battery lasts all day",
        "the case fits my screen perfectly", "i bought a screen protector for my screen",
        "the headphones were a bit over my budget but worth it",
        "i stayed within my budget and got a great phone",
        "this was a great purchase and i would buy it again",
        "i would not recommend this product to my friends",
        "the camera takes sharp photos in good light", "the camera is bad in low light",
        "the app is easy to use and the setup was quick",
        "i returned it because the screen cracked after two days",
        "my old phone was slower and the screen was dull",
        "the quality is fantastic and the price is fair",
        "the sound is fantastic and the bass is deep", "the speaker is loud and the sound is clear",
        "the watch looks good and the strap is comfortable", "i am very happy with this purchase",
        "the tablet is light and the screen is bright",
        "customer service replied quickly and solved my problem",
        "the mouse is smooth and the buttons are quiet",
        "the box was damaged but the product was fine",
        "the fan is noisy and the laptop gets hot", "great value for the money and fast shipping",
        "the keyboard feels nice and the keys are responsive",
    ]
    reviews = list(extras)
    for s in subjects:
        for p in random.sample(predicates, 8):
            reviews.append(f"{s} {p}")
    random.shuffle(reviews)
    with open("product_reviews_corpus.txt", "w") as f:
        f.write("\n".join(reviews))

In [3]:
# Load and tokenize the corpus (lowercased)
with open("product_reviews_corpus.txt", "r") as f:
    corpus = [line.strip().lower() for line in f if line.strip()]
tokenized_corpus = [word_tokenize(sentence) for sentence in corpus]

# Held-out sentence (lowercased in the same place as the training text)
held_out = "the movie was fantastic and the acting was convincing"
tokenized_held_out = word_tokenize(held_out.lower())

train_vocab = set(w for sent in tokenized_corpus for w in sent)
print("Reviews in corpus:", len(corpus))
print("Training vocabulary size:", len(train_vocab))
print("Held-out tokens:", tokenized_held_out)

Reviews in corpus: 142
Training vocabulary size: 141
Held-out tokens: ['the', 'movie', 'was', 'fantastic', 'and', 'the', 'acting', 'was', 'convincing']


In [4]:
def get_ngrams(tokens, order):
    """Pad a token list and return its n-grams in the format the model expects."""
    return list(ngrams(list(pad_both_ends(tokens, n=order)), order))

def build_model(model_class, order, **kwargs):
    """The pipeline iterator is consumed by fit(), so it is regenerated for every model."""
    train_data, vocab = padded_everygram_pipeline(order, tokenized_corpus)
    model = model_class(order=order, **kwargs)
    model.fit(train_data, vocab)
    return model

def sentence_prob(tokens, model, order):
    """Product of P(w_t | context) over all padded n-grams of the sentence."""
    prob = 1.0
    for ng in get_ngrams(tokens, order):
        prob *= model.score(ng[-1], list(ng[:-1]))
    return prob

held_out_bigrams = get_ngrams(tokenized_held_out, 2)
print(held_out_bigrams)

[('<s>', 'the'), ('the', 'movie'), ('movie', 'was'), ('was', 'fantastic'), ('fantastic', 'and'), ('and', 'the'), ('the', 'acting'), ('acting', 'was'), ('was', 'convincing'), ('convincing', '</s>')]


## Task 01: Two Smoothing Methods on the Bigram Model

In [5]:
# (a) Add-k (k=0.5) and Kneser-Ney, both on a bigram model
addk_bigram = build_model(Lidstone, 2, gamma=0.5)
kn_bigram = build_model(KneserNeyInterpolated, 2)

addk_ppl = addk_bigram.perplexity(held_out_bigrams)
kn_ppl = kn_bigram.perplexity(held_out_bigrams)

print(f"Add-k (k=0.5) perplexity:  {addk_ppl}")
print(f"Kneser-Ney perplexity:     {kn_ppl}")

Add-k (k=0.5) perplexity:  79.22024106995471
Kneser-Ney perplexity:     inf


In [6]:
# (b) which method is better and why
oov_words = [w for w in tokenized_held_out if w not in train_vocab]
print("Words in the held-out sentence that never occur in training:", oov_words)

print("\nProbability of every held-out bigram:")
rows = []
for bg in held_out_bigrams:
    rows.append({"bigram": bg,
                 "Add-k": addk_bigram.score(bg[1], [bg[0]]),
                 "Kneser-Ney": kn_bigram.score(bg[1], [bg[0]])})
print(pd.DataFrame(rows).to_string(index=False))

better = "Add-k (k=0.5)" if addk_ppl < kn_ppl else "Kneser-Ney"
print(f"\nBetter method on this held-out sentence: {better}")
print(f"Add-k = {addk_ppl:.2f}, Kneser-Ney = {kn_ppl}")
if kn_ppl == float("inf"):
    print("Kneser-Ney gives inf because the held-out sentence has out-of-vocabulary words "
          f"{oov_words}. NLTK maps them to <UNK>, which has a zero count in training, and the "
          "interpolation only shares probability mass between words that were seen. Add-k adds k "
          "to every count (including <UNK>), so every bigram gets a small non-zero probability "
          "and the perplexity stays finite.")

Words in the held-out sentence that never occur in training: ['movie', 'acting', 'convincing']

Probability of every held-out bigram:
            bigram    Add-k  Kneser-Ney
        (<s>, the) 0.577103    0.865574
      (the, movie) 0.002222    0.000000
      (movie, was) 0.006944    0.045016
  (was, fantastic) 0.005000    0.000126
  (fantastic, and) 0.031250    0.238867
        (and, the) 0.137755    0.496599
     (the, acting) 0.002222    0.000000
     (acting, was) 0.006944    0.045016
 (was, convincing) 0.005000    0.000000
(convincing, </s>) 0.006944    0.118971

Better method on this held-out sentence: Add-k (k=0.5)
Add-k = 79.22, Kneser-Ney = inf
Kneser-Ney gives inf because the held-out sentence has out-of-vocabulary words ['movie', 'acting', 'convincing']. NLTK maps them to <UNK>, which has a zero count in training, and the interpolation only shares probability mass between words that were seen. Add-k adds k to every count (including <UNK>), so every bigram gets a small non-ze

In [7]:
# (c) sanity check on a sentence that has no unseen words: here the interpolated models are usable
in_vocab_sentence = word_tokenize("the camera quality is excellent and the price is fair")
print("Any OOV words:", [w for w in in_vocab_sentence if w not in train_vocab])
check_ngrams = get_ngrams(in_vocab_sentence, 2)
print(f"Add-k (k=0.5) perplexity: {addk_bigram.perplexity(check_ngrams):.2f}")
print(f"Kneser-Ney perplexity:    {kn_bigram.perplexity(check_ngrams):.2f}")

Any OOV words: []
Add-k (k=0.5) perplexity: 17.57
Kneser-Ney perplexity:    7.35


In [8]:
# keep the winner so the later tasks do not depend on a hardcoded choice
methods = {
    "Add-k (k=0.5)": (Lidstone, {"gamma": 0.5}, addk_ppl),
    "Kneser-Ney": (KneserNeyInterpolated, {}, kn_ppl),
}
best_name = min(methods, key=lambda name: methods[name][2])
best_class, best_kwargs, _ = methods[best_name]
best_bigram = build_model(best_class, 2, **best_kwargs)
print("Best smoothing method used in the next tasks:", best_name)

Best smoothing method used in the next tasks: Add-k (k=0.5)


## Task 02: Next-Word Predictor for a Review-Writing Assistant

In [9]:
# (a) score each candidate with the best bigram model
partial = "the camera quality exceeded my"
candidates = ["expectations", "budget", "screen"]
context_word = word_tokenize(partial)[-1]

rows = []
for w in candidates:
    rows.append({
        "candidate": w,
        "P(word | 'my')": best_bigram.score(w, [context_word]),
        "full sentence prob": sentence_prob(word_tokenize(partial + " " + w), best_bigram, 2),
    })
ranking = pd.DataFrame(rows).sort_values("P(word | 'my')", ascending=False).reset_index(drop=True)
ranking.index = ranking.index + 1
ranking.index.name = "rank"
print(f"Model: {best_name} bigram\n")
print(ranking.to_string(float_format=lambda x: f"{x:.6g}"))

Model: Add-k (k=0.5) bigram

         candidate  P(word | 'my')  full sentence prob
rank                                                  
1     expectations        0.117978         2.68581e-07
2           budget       0.0280899         3.72954e-09
3           screen       0.0280899         9.51675e-09


In [10]:
# (b) what the assistant would suggest first
top = ranking.iloc[0]["candidate"]
print(f"Suggested first: '{partial} {top}'")
print(f"Order shown to the user: {list(ranking['candidate'])}")
# the full sentence probability also includes P(</s> | word), the ranking by the next-word
# probability is the one a keyboard uses because it only looks at what comes after 'my'

Suggested first: 'the camera quality exceeded my expectations'
Order shown to the user: ['expectations', 'budget', 'screen']


## Task 03: Bigram vs Trigram with the Chosen Smoothing

In [11]:
# (a) same smoothing method, order 2 and order 3
bigram_model = build_model(best_class, 2, **best_kwargs)
trigram_model = build_model(best_class, 3, **best_kwargs)

bigram_ppl = bigram_model.perplexity(get_ngrams(tokenized_held_out, 2))
trigram_ppl = trigram_model.perplexity(get_ngrams(tokenized_held_out, 3))

print(f"Smoothing method: {best_name}")
print(f"Bigram perplexity:  {bigram_ppl:.2f}")
print(f"Trigram perplexity: {trigram_ppl:.2f}")

Smoothing method: Add-k (k=0.5)
Bigram perplexity:  79.22
Trigram perplexity: 92.74


In [12]:
# (b) which one generalizes better
winner = "bigram" if bigram_ppl < trigram_ppl else "trigram"
print(f"The {winner} model generalizes better to the unseen sentence "
      f"(bigram {bigram_ppl:.2f} vs trigram {trigram_ppl:.2f}).")
print("A trigram context is two words long, so far fewer trigrams from the held-out sentence "
      "were ever seen in training than bigrams. Most trigram probabilities come from the "
      "smoothing constant alone, which spreads the mass more thinly over the larger set of "
      "possible events, and this raises the perplexity.")

The bigram model generalizes better to the unseen sentence (bigram 79.22 vs trigram 92.74).
A trigram context is two words long, so far fewer trigrams from the held-out sentence were ever seen in training than bigrams. Most trigram probabilities come from the smoothing constant alone, which spreads the mass more thinly over the larger set of possible events, and this raises the perplexity.


## Task 04: Out-of-Vocabulary Word, Unsmoothed vs Smoothed

In [13]:
# (a) find a word of the held-out sentence that is not in the training data
oov_words = [w for w in tokenized_held_out if w not in train_vocab]
oov_word = oov_words[0]
print("All OOV words:", oov_words)
print("Chosen OOV word:", repr(oov_word))
print("Count of the word in training data:", sum(s.count(oov_word) for s in tokenized_corpus))

All OOV words: ['movie', 'acting', 'convincing']
Chosen OOV word: 'movie'
Count of the word in training data: 0


In [14]:
# (b) unsmoothed model vs best smoothed model
mle_bigram = build_model(MLE, 2)

print(f"{'bigram':<22}{'MLE':>12}{best_name:>20}")
for bg in held_out_bigrams:
    if oov_word in bg:
        print(f"{str(bg):<22}{mle_bigram.score(bg[1], [bg[0]]):>12.6f}"
              f"{best_bigram.score(bg[1], [bg[0]]):>20.8f}")

bigram                         MLE       Add-k (k=0.5)
('the', 'movie')          0.000000          0.00222222
('movie', 'was')          0.000000          0.00694444


In [15]:
# (c) probability and perplexity of the whole held-out sentence
mle_prob = sentence_prob(tokenized_held_out, mle_bigram, 2)
smooth_prob = sentence_prob(tokenized_held_out, best_bigram, 2)
mle_ppl = mle_bigram.perplexity(held_out_bigrams)
smooth_ppl = best_bigram.perplexity(held_out_bigrams)

summary = pd.DataFrame({
    "Model": ["Unsmoothed (MLE)", f"{best_name}"],
    "Sentence probability": [f"{mle_prob}", f"{smooth_prob:.3e}"],
    "Perplexity": [mle_ppl, round(smooth_ppl, 2)],
})
print(summary.to_string(index=False))

print(f"\nThe unsmoothed model gives probability {mle_prob} (perplexity {mle_ppl}) because the OOV "
      f"word '{oov_word}' has a zero count.")
print(f"The {best_name} model gives probability {smooth_prob:.3e} and a finite perplexity of {smooth_ppl:.2f}.")

           Model Sentence probability  Perplexity
Unsmoothed (MLE)                  0.0         inf
   Add-k (k=0.5)            1.027e-19       79.22

The unsmoothed model gives probability 0.0 (perplexity inf) because the OOV word 'movie' has a zero count.
The Add-k (k=0.5) model gives probability 1.027e-19 and a finite perplexity of 79.22.
